# Inference: tree ensemble (from the Model Registry)

Loads the blended tree model **strictly from the DagsHub Model Registry**
(`Walmart_Tree_Ensemble`) and runs it on the **raw** `test.csv` to produce a Kaggle
submission. There is no preprocessing here — the registered pyfunc bundles both
pipelines and builds every feature itself.

Register the model first with `register_tree_ensemble.ipynb`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
import dagshub
from mlflow.tracking import MlflowClient

warnings.filterwarnings('ignore')
dagshub.init(repo_owner='slosa23', repo_name='Walmart-Sales-Forecasting', mlflow=True)
REGISTERED_NAME = 'Walmart_Tree_Ensemble'

client = MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED_NAME}'")
latest = max(int(v.version) for v in versions)
print('Loading', REGISTERED_NAME, 'version', latest)
model = mlflow.pyfunc.load_model(f'models:/{REGISTERED_NAME}/{latest}')

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

Loading Walmart_Tree_Ensemble version 1


## Predict on the raw test set and write the submission

In [3]:
test_raw = pd.read_csv('data/test.csv')
preds = model.predict(test_raw[['Store', 'Dept', 'Date', 'IsHoliday']])

submission = pd.DataFrame({
    'Id': test_raw['Store'].astype(str) + '_' + test_raw['Dept'].astype(str) + '_' + test_raw['Date'].astype(str),
    'Weekly_Sales': np.asarray(preds, dtype=float),
})
submission.to_csv('submission_tree_ensemble.csv', index=False)
print('rows:', len(submission),
      '| NaNs:', int(submission['Weekly_Sales'].isna().sum()),
      '| negatives:', int((submission['Weekly_Sales'] < 0).sum()))
submission.head()

rows: 115064 | NaNs: 0 | negatives: 0


,Id,Weekly_Sales
0,1_1_2012-11-02,35883.270411
1,1_1_2012-11-09,18963.630744
2,1_1_2012-11-16,20222.889820
3,1_1_2012-11-23,19534.432867
4,1_1_2012-11-30,24364.005326


## Notes

- The model is loaded from the registry (not a local run) and runs on raw `test.csv`.
- All feature engineering and the 0.45/0.55 XGBoost/LightGBM blend happen inside the
  registered pyfunc.
- Upload `submission_tree_ensemble.csv` to Kaggle.